In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder

In [2]:
data = pd.read_csv('data_cleaned/cleaned_heart_attack_prediction.csv')
data.head()

,age,gender,region,income_level,hypertension,diabetes,cholesterol_level,obesity,waist_circumference,family_history,...,blood_pressure_diastolic,fasting_blood_sugar,cholesterol_hdl,cholesterol_ldl,triglycerides,EKG_results,previous_heart_disease,medication_usage,participated_in_free_screening,heart_attack
0,60,Male,Rural,Middle,0,1,211,0,83,0,...,62,173,48,121,101,Normal,0,0,0,0
1,53,Female,Urban,Low,0,0,208,0,106,1,...,76,70,58,83,138,Normal,1,0,1,0
2,62,Female,Urban,Low,0,0,231,1,112,1,...,74,118,69,130,171,Abnormal,0,1,0,1
3,73,Male,Urban,Low,1,0,202,0,82,1,...,65,98,52,85,146,Normal,0,1,1,0
4,52,Male,Urban,Middle,1,0,232,0,89,0,...,75,104,59,127,139,Normal,1,0,1,1


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158355 entries, 0 to 158354
Data columns (total 28 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   age                             158355 non-null  int64  
 1   gender                          158355 non-null  object 
 2   region                          158355 non-null  object 
 3   income_level                    158355 non-null  object 
 4   hypertension                    158355 non-null  int64  
 5   diabetes                        158355 non-null  int64  
 6   cholesterol_level               158355 non-null  int64  
 7   obesity                         158355 non-null  int64  
 8   waist_circumference             158355 non-null  int64  
 9   family_history                  158355 non-null  int64  
 10  smoking_status                  158355 non-null  object 
 11  alcohol_consumption             158355 non-null  object 
 12  physical_activit

### Handle Categorical Features

In [4]:
categorical_features = [x for x in data.columns if data[x].dtype =='object']
categorical_features

['gender',
 'region',
 'income_level',
 'smoking_status',
 'alcohol_consumption',
 'physical_activity',
 'dietary_habits',
 'air_pollution_exposure',
 'stress_level',
 'EKG_results']

In [5]:
# drop redundant features
data = data.drop(columns = ['region','income_level','air_pollution_exposure','participated_in_free_screening'])
data.head()

,age,gender,hypertension,diabetes,cholesterol_level,obesity,waist_circumference,family_history,smoking_status,alcohol_consumption,...,blood_pressure_systolic,blood_pressure_diastolic,fasting_blood_sugar,cholesterol_hdl,cholesterol_ldl,triglycerides,EKG_results,previous_heart_disease,medication_usage,heart_attack
0,60,Male,0,1,211,0,83,0,Never,Unknown,...,113,62,173,48,121,101,Normal,0,0,0
1,53,Female,0,0,208,0,106,1,Past,Unknown,...,132,76,70,58,83,138,Normal,1,0,0
2,62,Female,0,0,231,1,112,1,Past,Moderate,...,116,74,118,69,130,171,Abnormal,0,1,1
3,73,Male,1,0,202,0,82,1,Never,Moderate,...,136,65,98,52,85,146,Normal,0,1,0
4,52,Male,1,0,232,0,89,0,Current,Moderate,...,127,75,104,59,127,139,Normal,1,0,1


In [6]:
categorical_features = [x for x in data.columns if data[x].dtype =='object']
categorical_features

['gender',
 'smoking_status',
 'alcohol_consumption',
 'physical_activity',
 'dietary_habits',
 'stress_level',
 'EKG_results']

In [7]:
# count the instances of each class of categorical features
data["gender"].value_counts()

gender
Male      82243
Female    76112
Name: count, dtype: int64

In [8]:
data["smoking_status"].value_counts()

smoking_status
Never      79183
Current    39771
Past       39401
Name: count, dtype: int64

In [9]:
data["alcohol_consumption"].value_counts()

alcohol_consumption
Unknown     94848
Moderate    47725
High        15782
Name: count, dtype: int64

In [10]:
data["physical_activity"].value_counts()

physical_activity
Low         63417
Moderate    63027
High        31911
Name: count, dtype: int64

In [11]:
data["dietary_habits"].value_counts()

dietary_habits
Unhealthy    95030
Healthy      63325
Name: count, dtype: int64

In [12]:
data["stress_level"].value_counts()

stress_level
Moderate    79366
High        47359
Low         31630
Name: count, dtype: int64

In [13]:
data["EKG_results"].value_counts()

EKG_results
Normal      126914
Abnormal     31441
Name: count, dtype: int64

### Lable Encode Categorical Features

In [14]:
categorical_feature = [x for x in data.columns if data[x].dtype =='object']
categorical_feature

['gender',
 'smoking_status',
 'alcohol_consumption',
 'physical_activity',
 'dietary_habits',
 'stress_level',
 'EKG_results']

In [15]:

features = [
    'gender',
    'smoking_status',
    'alcohol_consumption',
    'physical_activity',
    'dietary_habits',
    'stress_level',
    'EKG_results'
]

# create a LabelEncoder instance
le = LabelEncoder()

# encode each feature column
for col in features:
    data[col] = le.fit_transform(data[col])

data.head()


,age,gender,hypertension,diabetes,cholesterol_level,obesity,waist_circumference,family_history,smoking_status,alcohol_consumption,...,blood_pressure_systolic,blood_pressure_diastolic,fasting_blood_sugar,cholesterol_hdl,cholesterol_ldl,triglycerides,EKG_results,previous_heart_disease,medication_usage,heart_attack
0,60,1,0,1,211,0,83,0,1,2,...,113,62,173,48,121,101,1,0,0,0
1,53,0,0,0,208,0,106,1,2,2,...,132,76,70,58,83,138,1,1,0,0
2,62,0,0,0,231,1,112,1,2,1,...,116,74,118,69,130,171,0,0,1,1
3,73,1,1,0,202,0,82,1,1,1,...,136,65,98,52,85,146,1,0,1,0
4,52,1,1,0,232,0,89,0,0,1,...,127,75,104,59,127,139,1,1,0,1


In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158355 entries, 0 to 158354
Data columns (total 24 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   age                       158355 non-null  int64  
 1   gender                    158355 non-null  int64  
 2   hypertension              158355 non-null  int64  
 3   diabetes                  158355 non-null  int64  
 4   cholesterol_level         158355 non-null  int64  
 5   obesity                   158355 non-null  int64  
 6   waist_circumference       158355 non-null  int64  
 7   family_history            158355 non-null  int64  
 8   smoking_status            158355 non-null  int64  
 9   alcohol_consumption       158355 non-null  int64  
 10  physical_activity         158355 non-null  int64  
 11  dietary_habits            158355 non-null  int64  
 12  stress_level              158355 non-null  int64  
 13  sleep_hours               158355 non-null  f

In [18]:
data.columns

Index(['age', 'gender', 'hypertension', 'diabetes', 'cholesterol_level',
       'obesity', 'waist_circumference', 'family_history', 'smoking_status',
       'alcohol_consumption', 'physical_activity', 'dietary_habits',
       'stress_level', 'sleep_hours', 'blood_pressure_systolic',
       'blood_pressure_diastolic', 'fasting_blood_sugar', 'cholesterol_hdl',
       'cholesterol_ldl', 'triglycerides', 'EKG_results',
       'previous_heart_disease', 'medication_usage', 'heart_attack'],
      dtype='object')

In [19]:
# Remove outliers
iso = IsolationForest(contamination=0.01, random_state=42)
yhat = iso.fit_predict(data.drop(columns=["heart_attack"]))  # exclude label column
data["Outlier"] = yhat
df_new = data[data["Outlier"] == 1].drop(columns=["Outlier"])
data.shape[0], df_new.shape[0]

(158355, 156771)

In [20]:
# add encoded dataset to data_encoded folder
df_new.to_csv("data_encoded/encoded_heart_attack_prediction.csv", index=False)
